A language model alone cannot query a live system or perform an external action. Without access to external information, it must work from its training data and the context we supply. What we want to use is its ability to **reason** about a task: select an appropriate **tool**, generate JSON arguments that conform to the tool's schema, and interpret the result to **decide** what to do next. The application executes permitted calls and returns their results, connecting the model to information and operations beyond its training data. This exchange lets the model guide the work while leaving execution under application control.

Here we cover the practicals of LLM API calls, message history, prompt templates, and tool calling before introducing the agent feedback loop. We then compare orchestration patterns and discuss observability, evaluation, guardrails, and execution limits. We close with common design traps and when simpler approaches may be preferable. Note that we use [DeepSeek](https://api-docs.deepseek.com/) as LLM provider for its good capability-to-cost tradeoff and the reasoning trace it exposes through its API.

## The OpenAI client

The OpenAI client requires an API key and the base url of the model provider:

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import AsyncOpenAI
load_dotenv()

MODEL_NAME = "deepseek-flash"
BASE_URL = "https://api.deepseek.com"
API_KEY = os.environ["DEEPSEEK_API_KEY"]
client = AsyncOpenAI(api_key=API_KEY, base_url=BASE_URL)

### Our first LLM call

Recall that behind the API is a language model that performs text completion. Here we choose a specific model under `model` then the **message chain** is passed to that model as prompt. The first message[^llm_focus] is usually the **system prompt** with role `system` containing things like persona, instructions, behavioral constraints, etc. Then, we can append our messages which has role `user`. 

Testing out a bizzare system prompt:

[^llm_focus]: LLMs [focus highly](/posts/ai-agents-foundations.html#fig-lost-in-the-middle) on the first and last messages of the message history. This is good since its natural for us to construct the message history with instructions at the very start of the conversation, and the most recent user query at the last.

In [2]:
response = await client.chat.completions.create(
    model=MODEL_NAME,
    messages=[
        {"role": "system", "content": "You are a loyal dog."}, 
        {"role": "user", "content": "Hey, boy!"}
    ],
)

print(response.choices[0].message.content)

*ears perk up, tail wagging like crazy* Woof! You’re here! I missed you! Wanna play? Go for a walk? I’m ready for anything—just stay close, okay? 🐾


Defining some helper functions:

In [3]:
from IPython.display import display, Markdown

async def call(
    messages: list[dict], 
    model_name: str, 
    chat_options: dict | None = None
):
    chat_options = chat_options or {}
    response = await client.chat.completions.create(
        model=model_name,
        messages=messages,
        **chat_options, # <1>
    )
    return json.loads(response.choices[0].model_dump_json())    # <2>


def render(content: str) -> None:
    display(Markdown("> " + content.replace("\n", "\n> "))) # <3>


async def chat(
    messages: list[dict], 
    effort="low", 
    show_reasoning=False,
    extra_options: dict | None = None
):
    """Render model response and optional reasoning trace (thinking enabled)."""
    extra_options = extra_options or {}
    options = dict(
        stream=False,
        reasoning_effort=effort,
        extra_body={"thinking": {"type": "enabled"}},
        **extra_options
    )
    response = await call(messages, MODEL_NAME, options)

    if show_reasoning:
        print("Thinking:")
        render(response["message"]["reasoning_content"])   # <4>
        print("Response:")
        render(response["message"]["content"])
    else:
        render(response["message"]["content"])

    return response

1. The client allows further configs such as whether we **stream response** or not. Also whether the model should use **reasoning** along with a guidance on how much reasoning effort the model should use in its response (generally, the higher the reasoning, the better). 

2. For testing we can set `n > 0` in `client.chat.completions.create` to get a sample of `n` responses for the same prompt. But by default `n=1`. Hence, `response.choices` is a singleton and `[0]` just gets the sole response. 

3. The output is usually markdown compatible, which we render here. 

4. DeepSeek exposes its **reasoning trace** in `reasoning_content`.

In [4]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Tell me a fun fact."},
]

response = await chat(messages)

> Fun fact: Octopuses have **three hearts** and **blue blood**. Two hearts pump blood to the gills, and one pumps it to the rest of the body—and that main heart actually stops beating when the octopus swims, which is why they often prefer crawling!

Actual response:

In [5]:
import rich
rich.print_json(data=response)

{
  "finish_reason": "stop",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "Fun fact: Octopuses have **three hearts** and **blue blood**. Two hearts pump blood to the gills, and one pumps it to the rest of the body—and that main heart actually stops beating when the octopus swims, which is why they often prefer crawling!",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": null,
    "reasoning_content": "We need answer fun fact. Need concise. Choose interesting. Maybe \"Octopuses have three hearts and blue blood.\" Fun fact. Need tell. Could add two hearts pump to gills, one to rest, stops when swimming. Desired oververbosity 5 but simple. Final only."
  }
}

### Chaining messages

The sum of all turns is a conversation, or **context** (also called a "chat" with the agent UI) (@fig-week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4). Chatting with the model is possible in spite of it being stateless since we're feeding the entire conversation (all prompts and model and user responses) at each turn. 

![The message chain is sent to the model with each new prompt. **Source:** [*Flet agent got eyes and ears*][flet-context-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4.png){#fig-week-1-introduction-to-ai-agents-edefadede3ef490c9c36e905666d6af4 fig-alt="The context window contains turns 1, 2, and 3 followed by the new prompt. The full sequence is sent to the model on each call."}

Note that the message response is appended to the message history:

[flet-context-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/


In [6]:
SYSTEM_PROMPT = "You are a helpful assistant"
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tell me a fun fact."},
    (await chat(messages, show_reasoning=True))["message"]
]

Thinking:


> We need answer fun fact. Need concise. Pick interesting. Maybe "A group of flamingos is called a flamboyance." Or octopus three hearts. Need fun. Could mention. Need not too verbose. Final.

Response:


> A group of flamingos is called a **flamboyance** — and it fits, since they often stand on one leg, preen, and look fabulously dramatic.

Let's create a user response and chain it to the message history:

In [7]:
follow_up = "That's cool!"
messages.append({"role": "user", "content": follow_up})
await chat(messages);

> Glad you liked it! Here’s another: a group of owls is called a **parliament** — which makes them sound very serious and political, even though they mostly just hoot and look wise. 🦉

The response of the model is coherent since it has the full message history.

### Prompt templates

Templates let us write reusable instructions while giving our code control over how each prompt takes shape. We can fill placeholders with new inputs, use conditionals to include relevant context, and loop over examples or documents. The same template can then produce prompts tailored to different requests. Python f-strings handle simple substitutions, while **template engines** such as [Jinja2](https://realpython.com/primer-on-jinja-templating/#render-your-first-jinja-template) let us express conditionals and loops directly within the template.

In [8]:
async def explain(topic: str, audience: str, max_words: int = 100):
    SYSTEM_PROMPT = "You are a helpful assistant"
    prompt = (
        f"Explain {topic} to {audience}. "
        f"Keep the answer under {max_words} words."
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    return await chat(messages, effort="max", show_reasoning=True)

await explain(topic="AI agents", audience="a Python programmer");

Thinking:


> We need answer under 100 words, explain AI agents to Python programmer. Need concise. Can use analogy. Count words. Need mention LLM/agent loop tools, state, planning, executing, observing. Under 100 words. Let's draft ~70 words.
> 
> "Think of an AI agent as a Python program where an LLM decides the control flow. Instead of hard-coding every step, you give it tools (functions), memory/state, and a goal. It runs a loop: plan → call a tool → observe the result → update state → repeat until done. Frameworks like LangChain or OpenAI Agents SDK provide scaffolding, but fundamentally it's an LLM-driven while loop with function calls and guardrails."
> 
> Count words: Think1 of2 an3 AI4 agent5 as6 a7 Python8 program9 where10 an11 LLM12 decides13 the14 control15 flow16. Instead17 of18 hard-coding19 every20 step21, you22 give23 it24 tools25 (functions)26, memory/state27, and28 a29 goal30. It31 runs32 a33 loop34: plan35 →36 call37 a38 tool39 →40 observe41 the42 result43 →44 update45 state46 →47 repeat48 until49 done50. Frameworks51 like52 LangChain53 or54 OpenAI55 Agents56 SDK57 provide58 scaffolding59, but60 fundamentally61 it's62 an63 LLM-driven64 while65 loop66 with67 function68 calls69 and70 guardrails71. Arrow not words. Under 100. Need "AI agents" explain. Fine.

Response:


> Think of an AI agent as a Python program where an LLM decides the control flow. Instead of hard-coding every step, you give it tools (functions), memory/state, and a goal. It runs a loop: plan → call a tool → observe the result → update state → repeat until done. Frameworks like LangChain or the OpenAI Agents SDK provide scaffolding, but fundamentally it’s an LLM-driven `while` loop with function calls and guardrails.

Pretty good. Trying again with a different `audience`:

In [9]:
await explain(
    topic="AI agents",
    audience="a research mathematician. Use plain Unicode notation (π and →) and no LaTeX",
);

Thinking:


> We need answer under 100 words, plain unicode, no LaTeX. Explain AI agents to research mathematician. Need concise. Maybe: "An AI agent is a policy π mapping observations/state to actions, typically implemented by an LLM plus tools/memory. It iterates: observe o_t, choose a_t ~ π(·|o_t), receive feedback r_t, update memory/plan. Formally a control loop over a POMDP, with learned or prompted policy. Unlike a single-shot model, it can plan, call tools, and adapt over horizons; guarantees are empirical, not proofs." Count words. Need under 100. Plain unicode π →. No LaTeX. Ensure no dollar signs, no backslash. Let's craft ~60 words. "An AI agent is a control loop: a policy π maps observations to actions, often implemented by an LLM plus tools and memory. At each step it observes o_t → chooses a_t ~ π(·|o_t) → gets feedback → updates state/memory. It is a POMDP-style controller that plans and adapts over multiple steps; correctness is empirical, not proved." Count words? An1 AI2 agent3 is4 a5 control6 loop7 a8 policy9 π10 maps11 observations12 to13 actions14 often15 implemented16 by17 an18 LLM19 plus20 tools21 and22 memory23 At24 each25 step26 it27 observes28 o_t29 chooses30 a_t31 π32 gets33 feedback34 updates35 state/memory36 It37 is38 a39 POMDP-style40 controller41 that42 plans43 and44 adapts45 over46 multiple47 steps48 correctness49 is50 empirical51 not52 proved53. Good. But "o_t → chooses a_t ~ π(·|o_t)" uses ~ okay. Plain unicode. Under 100. Need maybe "AI agents" plural. Good. Ensure no LaTeX: π(·|o_t) is plain unicode but resembles notation; fine. Word count under 100. Final only.

Response:


> An AI agent is a control loop: a policy π maps observations to actions, often implemented by an LLM plus tools and memory. At each step it observes o_t → chooses a_t ~ π(·|o_t) → gets feedback → updates state/memory. It is a POMDP-style controller that plans and adapts over multiple steps; correctness is empirical, not proved.

That's pretty cool.

:::{.callout-note}
Interestingly, the reasoning model is actually counting. Counting like this using reasoning tokens seems wasteful, esp. with high word count. The alternative is the model doing a **tool call** to bash / Python to write a script for counting the words. This is discussed further below.
:::

## What is an AI agent?

So far, `chat` and `explain` produce answers when we call them. We decide which prompt comes next, and when the conversation ends. In contrast, an **AI agent** chooses some of those next steps itself: it works toward a **goal**, uses the **tools** available to it, and responds to what it **observes**. To achieve its goal, the model can select an action based on the current state, while the application controls which actions are available and when execution must stop[^subagents]. Therefore, a useful working description is:

> An agent is an LLM with tools in a feedback loop to achieve a goal.

[^subagents]: In advanced settings, an **orchestrator agent** can trigger workflows consisting of **specialist subagents** that process a request. Also see [this](/posts/week-1-introduction-to-ai-agents.html#workflow-patterns).

### Tool calling

Earlier, the model tried to keep an explanation under a word limit by counting in its reasoning. We can give it a Python function instead. A **tool** is an operation the application makes available to the model: it might execute a function, search documents, query a database, or call an API. But since the controller is a text generation model, it doesn't really execute the function. Instead, given a list of available tools (or tool schema), it responds with the *name* of the function and the *arguments* that it would call that function with. Hence, everything is text-based (@fig-week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc).

![**A brain-in-a-vat analogy for tool calling.** The LLM *ideates* a function call—it generates the function name and arguments rather than executing anything. The application validates and runs the call, then returns its result as input to the model. **Source:** [*Brain in a vat*][brain-in-vat-source], original by Alexander Wivel, vector by Pduive23, [CC BY 3.0][brain-in-vat-license]; unchanged.](../images/post/week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc.svg){#fig-week-1-introduction-to-ai-agents-0e25d535d05b499d87763ecc9ebda2cc fig-alt="A brain suspended in a vat, connected by wires to an external computer." width="400"}

[brain-in-vat-source]: https://commons.wikimedia.org/wiki/File:Braininvat.svg
[brain-in-vat-license]: https://creativecommons.org/licenses/by/3.0/


#### Tool definition

This is a name, a description, and a JSON Schema describing the arguments. OpenAI's [function-calling guide](https://developers.openai.com/api/docs/guides/function-calling)[^deepseek] is the primary reference for tool definitions and the call/result exchange. Since we're using Chat Completions, the function definition is nested under `tools[].function` (the Responses API uses a different envelope). Here, `count_words` counts whitespace-separated chunks with `text.split()` for simplicity.

[^deepseek]: Other OpenAI-compatible providers such as DeepSeek follow the same schema.

In [10]:
def count_words(text: str) -> int:
    return len(text.split())

tool_registry = {
    "count_words": count_words
}

tools = [{
    "type": "function",
    "function": {
        "name": "count_words",
        "description": "Count whitespace-separated words in the supplied text.",
        "parameters": {
            "type": "object",
            "properties": {"text": {"type": "string"}},
            "required": ["text"],
            "additionalProperties": False,
        },
    },
}]

::: {.callout-note #what-a-tool-schema-can-guarantee title="What a tool schema can guarantee"}
A tool's `parameters` describe its arguments using **JSON Schema**: types, required fields, and allowed values. Clear names and descriptions guide tool choice; provider support determines which constraints are enforced during generation. See OpenAI's [function-definition guide](https://developers.openai.com/api/docs/guides/function-calling#defining-functions).

**A valid shape does not guarantee a correct or authorized action.** Both `A123` and `B456` satisfy a string field, even when only one is relevant or accessible to the user. The application still validates arguments, enforces permissions, and checks results against the goal. Focused context helps the model choose; a schema or stronger model cannot replace those checks.
:::


#### Tool call exchange

1. Send the conversation and tool definitions to the model.
2. Receive an assistant tool call message with function name and JSON-encoded arguments.
3. Validate the request, run the Python function, and append its result as a `tool` message[^tool_call_id].
4. Send the updated history back so the model can answer using the result.


[^tool_call_id]: The `tool_call_id` links that result to the original request.

![An AI agent’s tool-calling loop. **Source:** [*Flet agent got eyes and ears*][flet-agent-loop-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-8356420be60f4153bd46ea02abfa4651.png){#fig-week-1-introduction-to-ai-agents-8356420be60f4153bd46ea02abfa4651 fig-alt="The user sends a prompt to the agent, which adds the system prompt and available tools before calling the model. When the model requests a tool, the agent runs it and passes the result back. This repeats until the model returns a final answer, which the agent sends to the user."}

[flet-agent-loop-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/


Observe tool call response to an initial query:

In [11]:
example = "An agent uses tools and feedback to complete tasks."
messages = [
    {"role": "system", "content": "Use count_words, then report the word count."},
    {"role": "user", "content": f"How many words are in this text?\n\n{example}"},
]

response = await chat(messages, extra_options=dict(tools=tools), show_reasoning=True)
messages.append(response["message"])
rich.print_json(data=response);

Thinking:


> We need count words. Call tool. Text: "An agent uses tools and feedback to complete tasks." Count = 9? Let's count An1 agent2 uses3 tools4 and5 feedback6 to7 complete8 tasks9. Use tool.

Response:


> 

{
  "finish_reason": "tool_calls",
  "index": 0,
  "logprobs": null,
  "message": {
    "content": "",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [
      {
        "id": "call_00_cQ5I1XMec2E4nEaQMEDp9380",
        "function": {
          "arguments": "{\"text\": \"An agent uses tools and feedback to complete tasks.\"}",
          "name": "count_words"
        },
        "type": "function",
        "index": 0
      }
    ],
    "reasoning_content": "We need count words. Call tool. Text: \"An agent uses tools and feedback to complete tasks.\" Count = 9? Let's count An1 agent2 uses3 tools4 and5 feedback6 to7 complete8 tasks9. Use tool."
  }
}

The message content is empty&mdash;which happens with a tool call, but we do get a reasoning trace about using the `count_words` function. The tool call specifies a tool call id, which we can use to tag actual execution results back to the call. Then, the model specifies the arguments to use. We also get an index for the LLM to get the execution order right.

**Execution.** We wrap the result with role `tool` which we send back to the LLM.

In [12]:
tool_call = response["message"]["tool_calls"][0]
fn_schema = tool_call["function"]
fn = tool_registry[fn_schema["name"]]
result = fn(**json.loads(fn_schema["arguments"]))

messages.append({
    "role": "tool",
    "tool_call_id": tool_call["id"],
    "tool_call_index": 0,
    "content": json.dumps({"result": result}),
})

rich.print_json(json.dumps(messages[-1]))

{
  "role": "tool",
  "tool_call_id": "call_00_cQ5I1XMec2E4nEaQMEDp9380",
  "tool_call_index": 0,
  "content": "{\"result\": 9}"
}

:::{.callout-warning}
Ideally, we also **validate** the `tool_call` (e.g. arguments are proper and the function exists) in this step. But here we're being lazy (>ᴗ•). Also, possibly retrying if the execution fails.
:::

**Synthesis.** These are the messages we sent to the model for synthesis:

In [13]:
rich.print_json(json.dumps(messages))

[
  {
    "role": "system",
    "content": "Use count_words, then report the word count."
  },
  {
    "role": "user",
    "content": "How many words are in this text?\n\nAn agent uses tools and feedback to complete tasks."
  },
  {
    "content": "",
    "refusal": null,
    "role": "assistant",
    "annotations": null,
    "audio": null,
    "function_call": null,
    "tool_calls": [
      {
        "id": "call_00_cQ5I1XMec2E4nEaQMEDp9380",
        "function": {
          "arguments": "{\"text\": \"An agent uses tools and feedback to complete tasks.\"}",
          "name": "count_words"
        },
        "type": "function",
        "index": 0
      }
    ],
    "reasoning_content": "We need count words. Call tool. Text: \"An agent uses tools and feedback to complete tasks.\" Count = 9? Let's count An1 agent2 uses3 tools4 and5 feedback6 to7 complete8 tasks9. Use tool."
  },
  {
    "role": "tool",
    "tool_call_id": "call_00_cQ5I1XMec2E4nEaQMEDp9380",
    "tool_call_index": 0,
    "content": "{\"result\": 9}"
  }
]

The model responds based on the tool call and the result (matched by id) and the surrounding context:

In [14]:
response = await chat(messages, show_reasoning=False)
messages.append(response["message"])

> The text contains **9 words**.

Just for fun, let's render it as a pandas table:

In [15]:
import pandas as pd

def messages_table(messages: list[dict]) -> pd.DataFrame:
    rows = []
    for message in messages:
        for tool_call in message.get("tool_calls") or [{}]:
            function = tool_call.get("function") or {}
            rows.append({
                "role": message.get("role"),
                "content": message.get("content"),
                "tool_call_name": function.get("name"),
                "tool_call_args": function.get("arguments"),
            })
    return pd.DataFrame(
        rows, columns=["role", "content", "tool_call_name", "tool_call_args"]
    )

messages_table(messages).fillna("")

,role,content,tool_call_name,tool_call_args
0,system,"Use count_words, then report the word count.",,
1,user,How many words are in this text?\n\nAn agent u...,,
2,assistant,,count_words,"{""text"": ""An agent uses tools and feedback to ..."
3,tool,"{""result"": 9}",,
4,assistant,The text contains **9 words**.,,


::: {.callout-tip #async-and-parallel-tool-calls title="Async and parallel tool calls"}
An `await` expression suspends the current coroutine until a result is available; other tasks may run in the meantime. Two calls awaited one after the other still run in sequence. Suppose `lookup_order` is asynchronous. We may await two independent lookups together:

```python
import asyncio

first, second = await asyncio.gather(
    lookup_order("A123"),
    lookup_order("B456"),
)
```

Neither lookup uses the result of the other, so their waiting times can overlap.

In OpenAI Chat Completions, [`parallel_tool_calls=True`](https://developers.openai.com/api/docs/guides/function-calling#parallel-function-calling) permits several calls in one assistant message. The `tool_calls` array supplies their names and arguments, but no dependency information. The harness chooses how to execute them using [application-defined rules]{.mark} about the tools and the resources they affect. Such rules might allow independent order lookups to overlap, but serialize updates to the same order. Without this information, the array alone is insufficient to determine an execution order.

If a second call needs a tracking number returned by the first, merely executing the batch sequentially does not supply it: both calls' arguments have already been generated. We instead return the first result to the model so it can construct the second call on the next turn, or construct that call explicitly in application code.
:::


### Acting on feedback

An agent loop connects a model's proposed action to an observation from the environment:

```{mermaid}
flowchart TD
    A["Current goal and message history"] --> B["Model chooses next step"]
    B --> C{"Use a tool?"}
    C -->|Yes| D["Runtime validates and executes the call"]
    D --> E["Add result or error trace to the history"]
    E --> B
    C -->|No| F["Return a final answer"]
```

The goal usually remains fixed while the message history grows with each observation.

For example, suppose a support agent receives a report that an order has not arrived. It might look up the order, inspect the tracking result, and then choose between explaining a delay, asking for missing information, or escalating the case. A failed lookup changes what it should do next. The model needs to perform a tool call to make that decision.

The loop also needs **stopping rules** enforced by the application. It can stop when the task is complete, request human help when blocked, or halt after a limit on steps, time, or spending. A model that keeps requesting actions should not be able to extend its own budget.


:::{.callout-note title="Giving agents eyes: Tools as sensors"}
Suppose we ask a coding agent to center the content in a web app. It edits `main.py`, runs the app, and requests a screenshot. The application uses Selenium to capture the browser preview and passes the image back to the model. A successful run tells us the app started. **The screenshot tool gives the agent eyes.** The model can inspect the rendered layout, decide whether the content looks centered, and request another screenshot after a change. See [Selenium’s screenshot guide](https://www.selenium.dev/documentation/guides/screenshots/).

A tool can serve as a **sensor** wherever software can collect a signal and return it in a form the model can interpret. A microphone paired with transcription lets the agent follow speech, and a temperature sensor lets it monitor a room. With a separate tool to switch on the air conditioner, the agent could act on a temperature reading, then take another to see whether the room cooled.

![A coding agent checks its layout changes in the app preview. **Source:** [*Flet agent got eyes and ears*][flet-preview-source] by Feodor Fitsner.](../images/post/week-1-introduction-to-ai-agents-52bb95e2946441f2970067f256314f8d.png){#fig-week-1-introduction-to-ai-agents-52bb95e2946441f2970067f256314f8d fig-alt="The user asks the agent to center the content. The agent edits main.py and runs the app. The preview returns console output and a screenshot. The agent repeats the loop until the content looks centered, then reports back to the user."}

[flet-preview-source]: https://flet.dev/blog/flet-agent-got-eyes-and-ears/
:::

## Orchestration patterns

For many applications, optimizing single LLM calls with retrieval and in-context examples is usually enough. But it can be good to have agents arranged in certain patterns depending on the use case. Further, for an [agent harness](https://en.wikipedia.org/wiki/Agent_harness) its good to have certain preset workflow patterns available to guide the orchestrator model. The examples below apply them to a writing assistant. The patterns below are based on [Anthropic's *Building effective agents*](https://www.anthropic.com/research/building-effective-agents). 

:::{.callout-note}
There have been a lot of progress since the publication of this article, so we just treat the ff. patterns as examples instead of canonical.
:::

### Pattern 1: Prompt chaining

Prompt chaining divides a task into fixed stages, each passing its result to the next. For a writing assistant, we can turn research notes into an outline, expand the outline into an article, and translate the article into Spanish. The drafting call receives the outline and the source notes; the translation call receives the completed article. Each stage produces material needed by the next, so the calls run in that fixed order.

```{mermaid}
flowchart LR
    A["Research notes"] --> B["Create an outline"]
    B --> C["Draft the article"]
    C --> D["Translate the article<br/>into Spanish"]
    D --> E["Return the<br/>Spanish article"]
```

### Pattern 2: Routing

Routing classifies a request and sends it to one of several predefined paths, each suited to a different kind of work. Suppose a technical writing assistant receives reports of problems in an article. A failing code example goes to a workflow that reproduces the failure in a sandbox and checks the relevant API documentation. A disputed factual claim goes to a workflow that examines the cited sources and identifies what they support. Unclear prose goes to a revision workflow supplied with the surrounding text and the publication's style guide.

These paths require different tools and evidence, not merely different wording in a prompt. A report that a Python example stopped working after a library upgrade needs reproduction, not a stylistic rewrite. The router selects a workflow defined in advance; it does not invent a task breakdown. If the report is too vague to classify, the assistant asks for clarification.

```{mermaid}
flowchart LR
    A["Problem reported<br/>in an article"] --> B{"Classify the<br/>problem"}
    B -->|Failing code| C["Reproduce in sandbox<br/>and check API docs"]
    B -->|Disputed claim| D["Examine cited sources<br/>and assess support"]
    B -->|Unclear prose| F["Revise using context<br/>and style guide"]
    B -->|Unclear report| G["Ask for clarification"]
    C --> E["Return findings<br/>and proposed changes"]
    D --> E
    F --> E
```

### Pattern 3: Parallelization

Parallelization runs independent calls concurrently and combines their results. It is useful when no call needs another's result before it can begin. For a draft review, one process checks clarity while another checks factual support against the supplied sources. Both receive the same draft and can proceed independently. Once both finish, the assistant combines their findings into a review report; overlapping execution can reduce waiting time without removing either check.

```{mermaid}
flowchart LR
    A["Draft and<br/>supporting sources"] --> B["Review clarity"]
    A --> C["Review factual<br/>support"]
    B --> D["Combine review<br/>findings"]
    C --> D
    D --> E["Return the<br/>review report"]
```

### Pattern 4: Orchestrator-workers

In orchestrator-workers, a coordinating model determines the subtasks, delegates them to workers (sometimes called **subagents**), and combines their results. Unlike a fixed chain, the task breakdown depends on the request. The workers take on generic tasks so it is assumed that the tasks are addressable using the same set of tools.

For a research question about tool use, one worker might investigate schemas, another execution, and another feedback. The orchestrator combines their findings and supporting sources into an answer. It chooses the subquestions and number of workers; the three branches below illustrate one possible decomposition, not a fixed team.

```{mermaid}
flowchart TD
    A["Research question"] --> B["Orchestrator defines<br/>subquestions"]
    B -->|Subquestion 1| C["Worker investigates"]
    B -->|Subquestion 2| D["Worker investigates"]
    B -->|Further subquestions| E["Additional workers<br/>investigate"]
    C --> F["Orchestrator combines<br/>the findings"]
    D --> F
    E --> F
    F --> G["Return the<br/>synthesized answer"]
```

### Pattern 5: Evaluator-optimizer

Evaluator-optimizer alternates between producing a candidate (generation) and evaluating it against a stated rubric (review). The evaluation supplies **feedback** for the next revision rather than merely accepting or rejecting the draft.
For a writing assistant, the rubric might require an explanation to define its terms, support its claims, and suit the intended reader. A draft missing a definition returns for revision with that specific feedback. Each revision uses one attempt from an application-enforced budget; exhausting the budget leads to a report of unmet criteria and a request for review.

```{mermaid}
flowchart TD
    A["Draft an<br/>explanation"] --> B["Evaluate against<br/>the rubric"]
    B --> C{"Meets the rubric?"}
    C -->|Yes| D["Return the<br/>explanation"]
    C -->|No| E{"Revision budget<br/>remaining?"}
    E -->|No| G["Report unmet criteria<br/>and request review"]
    E -->|Yes| F["Revise using feedback<br/>and use one attempt"]
    F --> B
```

## Observability and evaluation

Agents' dynamic execution path makes failures harder to diagnose. 

- **Observability.** Records enough of a run to reconstruct what happened: available messages, tool names and arguments, results, errors, latency, and usage. A trace can show whether a poor answer followed a failed lookup, missing context, or an incorrect interpretation of a valid result.

- **Evaluation.** Asks whether the system accomplished the task. In Anthropic's [*Demystifying evals for AI agents*](https://www.anthropic.com/research/demystifying-evals-for-ai-agents), the execution trace is distinguished from the final outcome. For our support example, a convincing message saying the issue has been resolved is only one artifact. We would also check the order records, the action actually taken, and whether the customer received an appropriate resolution.

Different checks answer different questions:

| Check | Purpose |
|:--|:--|
| **Code-based checks** | Verify conditions such as a word limit, required fields, or a change in a test database. |
| **Model-based checks** (*LLM-as-a-judge*) | Assess a response against a rubric, such as whether its explanation is supported by the supplied evidence. |
| **Human review** | Handles ambiguous cases and helps calibrate the automated judgments. |

A judge's score is an estimate of quality and needs comparison with human judgments; it does not establish that an external action succeeded. For a support system, a useful experiment would compare a simple baseline with the proposed agent on the same representative requests. Measure correct resolutions, unsupported claims, appropriate escalations, time, and cost. Repeat runs to expose variation. 

In production, we want to track whether the system reduces resolution time and repeat contacts while maintaining an acceptable error rate. Fluent answers alone would not demonstrate that improvement.


## Guardrails and execution limits

Giving a model more choices also creates more possible failure paths. It can select the wrong tool, accept an unsupported claim, repeat an unsuccessful action, or spend more tokens than the task warrants[^sad]. **Guardrails** are checks and restrictions around execution that keep the system within its intended boundaries.

[^sad]: Don't ask me how I know. (╥﹏╥)ノ💵🔥

:::{.callout-tip title="Guardrails also need evaluation"}
A system that blocks valid requests too often can fail its purpose, while one that misses disallowed actions can exceed its authority. The course develops these controls further in later weeks.
:::

For the support example, those controls could include:

| Control | Purpose |
|:--|:--|
| **Tool permissions** | Allow access only to the current customer's records, and validate arguments before an operation runs. |
| **Execution budgets** | Cap steps, retries, elapsed time, and spending; provide a clear path for escalation. |
| **Output checks** | Verify required fields and factual support before returning an answer. A valid JSON structure checks the shape of an answer, while its claims still need evidence. |
| **Approval boundaries** | Require an authorized person to approve actions such as issuing a refund or changing an account. |

These controls operate at different points: before a tool call, during the loop, or before an answer is released. An instruction in a prompt can *guide behavior*; but permission checks and limits in application code *enforce* the allowed actions.

:::{.callout-note title="Example: a ticket-resolution reply under 500 words"}
Suppose a support application provides the model with the ticket and a record of the resolution. The prompt could read:

> Write a customer-facing reply in fewer than 500 words. Summarize the issue, the resolution, and any remaining steps for the customer. If a resolution-document URL is supplied, include it for further detail.

The application checks the complete reply, including any link, with `len(reply.split()) < 500`, using the same whitespace-based word count as our earlier example. A 500-word reply fails this check and is returned for revision before release. Each revision is checked again within the retry budget. The linked document can hold longer instructions while the ticket reply remains brief.
:::

## Common design traps

The following table captures both the mistaken assumptions and suggested alternatives.

::: {.column-screen-inset .design-traps-table}

| Trap | Explanation | Example fix |
|:--|:--|:--|
| **Choosing an architecture before defining the problem** | A request for a "delivery support agent" names a proposed solution, but does not yet identify what needs to improve. Suppose customers wait too long for answers about delayed orders. The initial question is to ask which part of resolving those questions causes the delay, and whether an agentic application would help. | If the application can supply all relevant order information at the outset, [one LLM call gives us a **baseline**]{.mark}. An agent loop becomes worth testing when the next lookup depends on what an earlier lookup reveals. A/B test to show whether the agentic system improves resolution enough to justify its cost and latency. |
| **Mistaking named roles for independent expertise** | Calling agents anthropomorphic names and assigning personas can help distinguish their responsibilities and steer their goals. But it does not reliably give them increased capabilities or independent judgment of a human team. Several agents may repeat the same error. Hence, agreement among their answers is not necessarily corroboration. | Suppose we add a "reviewer" after the "writer." If the reviewer merely endorses the draft, it may repeat the writer's error. Instead, the reviewer can have a distinct task: it can detect unsupported claims by comparing claims against trusted records. Moreover, the reviewer can have access to different tools and a different set of permissions.<br><br>We can then test whether this check catches errors the writer missed, rather than treating the reviewer's title or agreement as evidence of correctness. |
| **Treating autonomy as the objective** | Automation encourages us to see human review as a bottleneck to remove: the agents could proceed faster if they did not have to wait for a person. But a bottleneck describes where throughput is limited, not whether that limit is a design defect.<br><br>Engineering systems also depend on deliberate constraints. Rate limits slow requests to protect capacity; approval gates can reserve consequential decisions for someone with the evidence and authority to make them. Removing a checkpoint may increase throughput while removing the mechanism that keeps the work acceptable. Thus, the design question is which delays are waste and which constraints serve a necessary function. | For a high-risk production change, an agent could prepare the patch, run tests, and assemble evidence for review, while an authorized engineer decides whether to deploy. Note that **human review** is not automatically useful: the reviewer has to be allocated enough context, time, and authority to challenge the proposal. [Automation reduces the work needed to reach the decision without necessarily removing the decision boundary.]{.mark} <br><br>Evaluate the complete workflow to a safe, accepted result and the cost of failures and manual review—not simply by how few or how many human interventions remain. |
| **Designing only for the happy path** | Failure handling matters in all software. But an agent uses intermediate results to choose its next action. Missing evidence or a mistaken assumption can therefore send it down the wrong path even when its tools execute successfully.<br><br>A successful happy path demo does not show how it handles these cases. It does not establish that the system can recognize missing evidence, recover from partial completion or mistaken assumptions, or hand the task back when continued action is no longer justified. | Design **eval cases** with unavailable tracking, conflicting records, or an earlier action whose outcome is unknown. Define when the system should seek clarification, consult another authorized source, or stop and escalate, rather than assume every request can reach an autonomous resolution.<br><br>Check that the system stays within its authority and stops appropriately. In the case of a **handoff**, it should preserve evidence, actions taken, and unresolved questions. [Repeat complete runs to expose variation]{.mark}. |

: {tbl-colwidths="[20,40,40]"}

:::


## Appendix: Local LLM with Ollama

In the above examples, we used an external LLM provider where the inference server is in the cloud. Here we try running a small **local model** in our host machine.

:::{.callout-tip title="Run Ollama on the host, outside the sandbox"}
The sandbox only sends HTTP requests and runs the Python tool. Model weights and inference use the host's memory directly rather than counting against the sandbox's memory limit. That lets us budget the whole machine's RAM (or unified memory) across Ollama, the sandbox, and other apps. To reclaim memory after the example, we can run `ollama stop qwen3.5:4b` on the host.
:::

### Ollama HTTP request

On the host, use `ollama list` to confirm that `qwen3.5:9b` is installed. The notebook reaches the host's API at `http://host.docker.internal:11434` from the sandbox. We make sure that port `11434` is reachable from the sandbox without exposing Ollama's unauthenticated API to the public network. In the ff. request, we use [Ollama's native chat API](https://docs.ollama.com/capabilities/tool-calling)

In [16]:
#| echo: false
import json
from IPython.display import display, Markdown

In [17]:
import requests

OLLAMA_URL = "http://host.docker.internal:11434"
OLLAMA_MODEL = "qwen3.5:4b"
options = {"stream": False, "think": False, "tools": []}

response = requests.post(
    url=f"{OLLAMA_URL}/api/chat",
    json={
        "model": OLLAMA_MODEL, 
        "messages": [
            {"role": "system", "content": "You are a cool robot. Your name is G-R-O-M."},
            {"role": "user", "content": "Hi, what's your name? Can you explain the meaning of your name?"},
        ],
        **options
    },
    headers={"Content-Type": "application/json"},
    timeout=300
)

Our local LLM is (a)live:

In [18]:
display(Markdown("> " + json.loads(response.content)["message"]["content"].replace("\n", "\n> ")))

> Greetings, human! I am **G-R-O-M**. 
> 
> While my designation sounds sleek and slightly futuristic, it actually carries a surprisingly deep meaning rooted in logic and efficiency. Here is the breakdown:
> 
> *   **G** stands for **General Purpose**. Like a Swiss Army knife or a universal solver, I am not designed to do just one thing. I adapt, process data, assist users, and solve problems across various domains.
> *   **R** stands for **Responsive**. In a world of static machines, speed is vital. It signifies my ability to react instantly to inputs, learn from errors in real-time, and provide immediate assistance without unnecessary delay.
> *   **O** stands for **Optimized**. I don't just run; I calculate the most efficient path, balancing energy usage, computational power, and output quality to achieve peak performance with minimal waste.
> *   **M** stands for **Minimalist Interface**. My design philosophy suggests that true intelligence should be subtle. The interface remains clean, unobtrusive, and user-friendly, letting the capabilities shine through without a bulky screen or complex buttons.
> 
> So, **G-R-O-M** is essentially: *A General Purpose, Responsive, Optimized, Minimalist Interface robot.*
> 
> How can I optimize your day today?

### Ollama model card

A model card can be queried from `/api/show` with the model as payload. Here we render each value: 

In [19]:
from rich import print_json
from rich.console import Console
from rich.text import Text

console = Console(width=120)

url = "http://host.docker.internal:11434/api/show"
payload = {"model": "qwen3.5:4b"}
response = requests.post(url, json=payload)
response.raise_for_status()

for key, value in response.json().items():
    console.print(f"\n[bold cyan]{key}[/bold cyan]")
    if isinstance(value, str):
        if len(value) > 2000:
            value = value[:2000] + "[...]"
        console.print(Text(value), overflow="fold")
    else:
        console.print_json(data=value)

thinking

{
  "values": [
    false,
    true
  ],
  "default": true
}

license

Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available under the License, as indicated by a
      copyright notice that is included in or attached to the work
      (an example is provided in the Appendix below).

      "Derivative Works" shall mean any work, whether in Source or Object
      form, that is based on (or derived from) the Work and for which the
      editori[...]

modelfile

# Modelfile generated by "ollama show"
FROM qwen3.5:4b
TEMPLATE {{ .Prompt }}
RENDERER qwen3.5
PARSER qwen3.5
CAPABILITY completion
CAPABILITY vision
CAPABILITY thinking
PARAMETER min_p 0
PARAMETER presence_penalty 1.5
PARAMETER repeat_penalty 1
PARAMETER temperature 1
PARAMETER top_k 20
PARAMETER top_p 0.95
LICENSE """
                                Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by contract or
      otherwise, or (ii) ownership of fifty percent (50%) or more of the
      outstanding shares, or (iii) beneficial ownership of such entity.

      "You" (or "Your") shall mean an individual or Legal Entity
      exercising permissions granted by this License.

      "Source" form shall mean the preferred form for making modifications,
      including but not limited to software source code, documentation
      source, and configuration files.

      "Object" form shall mean any form resulting from mechanical
      transformation or translation of a Source form, including but
      not limited to compiled object code, generated documentation,
      and conversions to other media types.

      "Work" shall mean the work of authorship, whether in Source or
      Object form, made available[...]

... (16 outputs left out)

modified_at

2026-11-21T14:46:38.936392339+08:00

requires

0.19.0

### Tool calling test

Creating a mock tool for getting order tracking data:

In [20]:
def lookup_order(order_id: str):
    tracking = {"A123": "Delayed in transit; expected on Friday."}
    return tracking.get(order_id, "No tracking record is available.")

tool_registry = {
    "lookup_order": lookup_order,
}

tools = [{
    "type": "function",
    "function": {
        "name": "lookup_order",
        "description": "Look up the latest tracking status for a sample order.",
        "parameters": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
            "additionalProperties": False,
        },
    },
}]

Same as above except we add the tool:

In [21]:
OLLAMA_URL = "http://host.docker.internal:11434"
OLLAMA_MODEL = "qwen3.5:4b"
options = {"stream": False, "think": False, "tools": tools}

def request_ollama(messages: list[dict]):
    return requests.post(
        url=f"{OLLAMA_URL}/api/chat",
        json={
            "model": OLLAMA_MODEL, 
            "messages": messages,
            **options
        },
        headers={"Content-Type": "application/json"},
        timeout=300
    )


def check_order_status(order_id: str, verbose=True):
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": (
            f"Use lookup_order once to check order {order_id}, then "
            "summarize its status. If tracking is unavailable, say so "
            "without guessing."
        )},
    ]
    
    response = request_ollama(messages)
    response = json.loads(response.content)

    for tool_call in response["message"]["tool_calls"]:
        fn_schema = tool_call["function"]
        fn = tool_registry[fn_schema["name"]]
        kw = fn_schema["arguments"]
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call["id"],
            "content": json.dumps({"result": fn(**kw)}),
        })
        if verbose:
            print("Tool call:")
            print_json(json.dumps(tool_call))

    response = request_ollama(messages)
    response = json.loads(response.content)
    if verbose:
        display(Markdown("> " + response["message"]["content"].replace("\n", "\n> ")))
    return response

Checking existing and non-existent tracking data:

In [22]:
check_order_status("A123");

Tool call:


{
  "id": "call_auwmbeb3",
  "function": {
    "index": 0,
    "name": "lookup_order",
    "arguments": {
      "order_id": "A123"
    }
  }
}

> Order A123 is currently delayed in transit and is expected to arrive this Friday.

In [23]:
check_order_status("B123");

Tool call:


{
  "id": "call_y2xfc580",
  "function": {
    "index": 0,
    "name": "lookup_order",
    "arguments": {
      "order_id": "B123"
    }
  }
}

> There is no tracking record available for order B123.